# Exploratory Data Analysis (EDA)
## UCI Bank Marketing Dataset

This notebook performs exploratory data analysis on the Bank Marketing dataset, which is used to predict whether a customer will subscribe to a term deposit (`y`).

### Tech Stack used for analysis:
- Pandas
- NumPy
- Matplotlib / Seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Load the Dataset
The dataset is comma-separated (or semicolon-separated in the raw file).

In [ ]:
df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")
print(f"Dataset Shape: {df.shape}")
df.head()

### 2. General Information & Missing Values
Check columns, data types, and null counts.

In [ ]:
df.info()

In [ ]:
print("Standard missing values (Nulls):")
print(df.isnull().sum())

print("\nChecking for 'unknown' values (custom missing code in Bank dataset):")
for col in df.columns:
    unknown_count = (df[col] == "unknown").sum()
    if unknown_count > 0:
        print(f"{col}: {unknown_count} unknown values ({unknown_count/len(df)*100:.2f}%)")

### 3. Target Variable Analysis (Class Imbalance)
Check class counts and percentages for subscription indicator `y`.

In [ ]:
y_counts = df["y"].value_counts(normalize=True) * 100
print(f"Target class distribution (%):\n{y_counts}")

sns.countplot(x="y", data=df)
plt.title("Subscription Class Distribution (y)")
plt.show()

### 4. Numerical Feature Distributions
Explore key numerical distributions like age and duration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df["age"], kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Distribution of Customer Age")

sns.histplot(df["duration"], kde=True, ax=axes[1], color="salmon")
axes[1].set_title("Distribution of Call Duration (seconds)")
axes[1].set_xlim(0, 2000)

plt.show()

### 5. Correlation of Numerical Features

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number])
corr = numeric_cols.corr()

sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Numeric Features")
plt.show()

### 6. Subscription Patterns by Categorical Features
Check how different jobs or contact methods affect subscription rate.

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(x="job", hue="y", data=df)
plt.title("Subscription status (y) by Job Type")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="contact", hue="y", data=df)
plt.title("Subscription status (y) by Contact Type")
plt.show()

### Key Takeaways from EDA:
1. **Imbalance**: High class imbalance (~88.7% 'no' vs. ~11.3% 'yes'). Evaluation metrics should focus on ROC-AUC and F1-score rather than simple Accuracy.
2. **Missing Values**: Standard python nulls do not exist, but 'unknown' serves as a category representing missing features (e.g. 20%+ for `default`). We keep these as a valid class during One-Hot encoding.
3. **Call Duration**: Extremely right-skewed. Subscribers tend to have longer conversations, which makes intuitive sense but is technically leaked information before a call. We keep it as features since it is part of the UCI dataset specification.
4. **Correlation**: High correlation is observed between indicators such as `euribor3m` and `nr.employed` (~0.95), and `emp.var.rate` and `euribor3m` (~0.97). Features should be scaled and managed correctly.